In [1]:
.appName("NYC Building Risk - Gold HPD Violations")

SyntaxError: invalid syntax (2334570002.py, line 1)

In [2]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# ==================================================
# PROJECT CONFIG
# ==================================================

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)


# ==================================================
# IMPORT PROJECT HELPERS
# ==================================================

from minio_config import configure_minio, minio_path


# ==================================================
# CREATE / GET SPARK SESSION
# ==================================================

spark = (
    SparkSession.builder
    .appName("NYC Building Risk - Gold HPD Violations")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)

configure_minio(spark)

spark.sparkContext.setLogLevel("WARN")


# ==================================================
# TEST
# ==================================================

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Test:", spark.range(1).count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/07 17:56:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/07 17:56:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/07 17:56:46 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/09/07 17:56:46 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Spark version: 3.4.0
Master: local[2]
Test: 1


In [3]:
# ==================================================
# LOAD HPD SILVER
# ==================================================

hpd_df = spark.read.parquet(
    minio_path("silver/hpd")
)

print("HPD rows:", hpd_df.count())

print("\nHPD SILVER SCHEMA")
hpd_df.printSchema()

26/09/07 17:57:02 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


HPD rows: 927308

HPD SILVER SCHEMA
root
 |-- violationid: string (nullable = true)
 |-- novid: string (nullable = true)
 |-- buildingid: string (nullable = true)
 |-- registrationid: string (nullable = true)
 |-- bbl: string (nullable = true)
 |-- bin: string (nullable = true)
 |-- boroid: string (nullable = true)
 |-- boro: string (nullable = true)
 |-- block: string (nullable = true)
 |-- lot: string (nullable = true)
 |-- housenumber: string (nullable = true)
 |-- lowhousenumber: string (nullable = true)
 |-- highhousenumber: string (nullable = true)
 |-- streetname: string (nullable = true)
 |-- apartment: string (nullable = true)
 |-- story: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- class: string (nullable = true)
 |-- violationstatus: string (nullable = true)
 |-- currentstatus: string (nullable = true)
 |-- currentstatusid: string (nullable = true)
 |-- inspectiondate: timestamp (nullable = true)
 |-- novissueddate: timestamp (nullable = true)
 |-- curren

In [4]:
# ==================================================
# HPD BASIC VALIDATION
# ==================================================

print(
    "Total HPD rows:",
    hpd_df.count()
)

print(
    "Distinct violationid:",
    hpd_df
    .select("violationid")
    .distinct()
    .count()
)

print(
    "Missing violationid:",
    hpd_df
    .filter(F.col("violationid").isNull())
    .count()
)

print(
    "With BIN:",
    hpd_df
    .filter(
        F.col("bin").isNotNull()
        & (F.trim(F.col("bin")) != "")
    )
    .count()
)

print(
    "Without BIN:",
    hpd_df
    .filter(
        F.col("bin").isNull()
        | (F.trim(F.col("bin")) == "")
    )
    .count()
)

Total HPD rows: 927308


Distinct violationid: 927308


Missing violationid: 0


With BIN: 926505


Without BIN: 803


In [5]:
# ==================================================
# LOAD DIM_BUILDING
# ==================================================

dim_building_df = spark.read.parquet(
    minio_path(
        "gold/data_model/dim_building"
    )
)

print(
    "dim_building rows:",
    dim_building_df.count()
)


# ==================================================
# HPD BIN -> DIM_BUILDING
# ==================================================

building_lookup = (
    dim_building_df
    .select(
        F.col("bin")
        .cast("string")
        .alias("source_bin"),

        "building_id",

        F.col("property_id")
        .alias("building_property_id")
    )
    .dropDuplicates(
        ["source_bin"]
    )
)


hpd_stage = (
    hpd_df

    .withColumn(
        "source_bin",
        F.col("bin").cast("string")
    )

    .withColumn(
        "source_bbl_original",
        F.col("bbl").cast("string")
    )

    .join(
        building_lookup,
        on="source_bin",
        how="left"
    )
)

dim_building rows: 197958


In [6]:
print(
    "HPD rows:",
    hpd_stage.count()
)

print(
    "With building_id:",
    hpd_stage
    .filter(
        F.col("building_id").isNotNull()
    )
    .count()
)

print(
    "Without building_id:",
    hpd_stage
    .filter(
        F.col("building_id").isNull()
    )
    .count()
)

print(
    "With canonical property_id:",
    hpd_stage
    .filter(
        F.col("building_property_id").isNotNull()
    )
    .count()
)

print(
    "Without canonical property_id:",
    hpd_stage
    .filter(
        F.col("building_property_id").isNull()
    )
    .count()
)

HPD rows: 927308


With building_id: 926505


Without building_id: 803


With canonical property_id: 925982


Without canonical property_id: 1326


In [7]:
# ==================================================
# LOAD DIM_PROPERTY
# ==================================================

dim_property_df = spark.read.parquet(
    minio_path(
        "gold/data_model/dim_property"
    )
)


# ==================================================
# HPD SOURCE BBL -> DIM_PROPERTY
# ==================================================

property_lookup = (
    dim_property_df
    .select(
        F.col("bbl")
        .cast("string")
        .alias("source_bbl_original"),

        F.col("property_id")
        .alias("source_property_id")
    )
    .dropDuplicates(
        ["source_bbl_original"]
    )
)


hpd_stage_v2 = (
    hpd_stage

    .join(
        property_lookup,
        on="source_bbl_original",
        how="left"
    )
)

In [8]:
both_property = (
    F.col("building_property_id").isNotNull()
    & F.col("source_property_id").isNotNull()
)

print(
    "Both Property IDs available:",
    hpd_stage_v2
    .filter(both_property)
    .count()
)

print(
    "Same Property:",
    hpd_stage_v2
    .filter(
        both_property
        & (
            F.col("building_property_id")
            == F.col("source_property_id")
        )
    )
    .count()
)

print(
    "Different Property:",
    hpd_stage_v2
    .filter(
        both_property
        & (
            F.col("building_property_id")
            != F.col("source_property_id")
        )
    )
    .count()
)

print(
    "Building Property only:",
    hpd_stage_v2
    .filter(
        F.col("building_property_id").isNotNull()
        & F.col("source_property_id").isNull()
    )
    .count()
)

print(
    "Source Property only:",
    hpd_stage_v2
    .filter(
        F.col("building_property_id").isNull()
        & F.col("source_property_id").isNotNull()
    )
    .count()
)

print(
    "No Property:",
    hpd_stage_v2
    .filter(
        F.col("building_property_id").isNull()
        & F.col("source_property_id").isNull()
    )
    .count()
)

Both Property IDs available: 925363


Same Property: 923368


Different Property: 1995


Building Property only: 619


Source Property only: 796


No Property: 530


In [9]:
# ==================================================
# FINAL PROPERTY RESOLUTION FOR HPD
# ==================================================

hpd_final_stage = (
    hpd_stage_v2

    .withColumn(
        "property_id",
        F.when(
            F.col("building_property_id").isNotNull(),
            F.col("building_property_id")
        )
        .when(
            F.col("building_id").isNull()
            & F.col("source_property_id").isNotNull(),
            F.col("source_property_id")
        )
    )

    .withColumn(
        "property_resolution_method",
        F.when(
            F.col("building_property_id").isNotNull(),
            F.lit("BUILDING_CANONICAL")
        )
        .when(
            F.col("building_id").isNull()
            & F.col("source_property_id").isNotNull(),
            F.lit("SOURCE_BBL")
        )
        .otherwise(
            F.lit("UNRESOLVED")
        )
    )

    .withColumn(
        "property_conflict_flag",
        F.when(
            F.col("building_property_id").isNotNull()
            & F.col("source_property_id").isNotNull()
            & (
                F.col("building_property_id")
                != F.col("source_property_id")
            ),
            F.lit(1)
        ).otherwise(F.lit(0))
    )

    .withColumn(
        "building_resolution_status",
        F.when(
            F.col("building_id").isNotNull(),
            F.lit("RESOLVED")
        ).otherwise(
            F.lit("UNRESOLVED")
        )
    )
)

In [10]:
print(
    "Total HPD rows:",
    hpd_final_stage.count()
)

print(
    "With building_id:",
    hpd_final_stage
    .filter(F.col("building_id").isNotNull())
    .count()
)

print(
    "With property_id:",
    hpd_final_stage
    .filter(F.col("property_id").isNotNull())
    .count()
)

print(
    "Without property_id:",
    hpd_final_stage
    .filter(F.col("property_id").isNull())
    .count()
)

print(
    "Property conflicts:",
    hpd_final_stage
    .filter(F.col("property_conflict_flag") == 1)
    .count()
)

(
    hpd_final_stage
    .groupBy("property_resolution_method")
    .count()
    .show(truncate=False)
)

Total HPD rows: 927308


With building_id: 926505


With property_id: 926778


Without property_id: 530


Property conflicts: 1995


+--------------------------+------+
|property_resolution_method|count |
+--------------------------+------+
|UNRESOLVED                |530   |
|SOURCE_BBL                |796   |
|BUILDING_CANONICAL        |925982|
+--------------------------+------+



In [11]:
# ==================================================
# FINAL FACT_HPD_VIOLATION
# Grain: 1 row = 1 HPD violation
# ==================================================

fact_hpd_violation = (
    hpd_final_stage

    .withColumn(
        "hpd_event_id",
        F.concat(
            F.lit("HPD:"),
            F.col("violationid")
        )
    )

    .select(
        # Keys
        "hpd_event_id",
        "violationid",
        "novid",

        "property_id",
        "building_id",

        # Source identity
        "source_bin",
        "source_bbl_original",
        "source_property_id",
        "building_property_id",

        "property_resolution_method",
        "property_conflict_flag",
        "building_resolution_status",

        # HPD attributes
        "buildingid",
        "registrationid",

        "class",
        "violationstatus",
        "currentstatus",
        "currentstatusid",

        "inspectiondate",
        "novissueddate",
        "currentstatusdate",
        "approveddate",
        "certifieddate",
        "originalcorrectbydate",
        "originalcertifybydate",

        "novtype",
        "novdescription",
        "rentimpairing",

        # Address
        "boroid",
        "boro",
        "block",
        "lot",
        "housenumber",
        "lowhousenumber",
        "highhousenumber",
        "streetname",
        "apartment",
        "story",
        "zip",

        "latitude",
        "longitude",
        "communityboard",
        "councildistrict",
        "censustract",
        "nta",

        # Original source metadata
        "bbl_source",

        # Calendar helpers
        "inspection_day",
        "inspection_year",
        "inspection_month"
    )
)

In [12]:
print(
    "Fact rows:",
    fact_hpd_violation.count()
)

print(
    "Distinct hpd_event_id:",
    fact_hpd_violation
    .select("hpd_event_id")
    .distinct()
    .count()
)

Fact rows: 927308


Distinct hpd_event_id: 927308


In [13]:
# ==================================================
# SAVE FACT_HPD_VIOLATION
# ==================================================

FACT_HPD_PATH = minio_path(
    "gold/data_model/fact_hpd_violation"
)

(
    fact_hpd_violation
    .write
    .mode("overwrite")
    .parquet(FACT_HPD_PATH)
)

print("fact_hpd_violation saved successfully")
print("Path:", FACT_HPD_PATH)

26/09/07 18:04:06 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


fact_hpd_violation saved successfully
Path: s3a://nyc-building-risk/gold/data_model/fact_hpd_violation
